# 후보 모델에 대해 하이퍼파라미터 선택
### 후보 모델
1. 로지스틱 회귀 + 특성 v5
2.

In [1]:
# 데이터셋 로드
import pandas as pd

train_origin = pd.read_csv("../data/processed/01/train.csv")

## 후보 1 테스트


In [2]:
from src.feature import prep_v5
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

prep = prep_v5()
model = make_pipeline(prep, LogisticRegression())

파라미터 조합 설정

In [3]:
param_grid = [{
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [1],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["liblinear"],
     "logisticregression__class_weight": [None, "balanced"]
},
    {
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [0],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["lbfgs"],
         "logisticregression__class_weight": [None, "balanced"]
},
]

하이퍼 파라미터 탐색 객체 생성

In [4]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=10,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)


학습 및 결과확인


In [5]:
train = train_origin.copy()
labels = train["Survived"]

grid_search.fit(train, labels)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'logisticregression__C': 1, 'logisticregression__class_weight': None, 'logisticregression__l1_ratio': 0, 'logisticregression__max_iter': 10000, 'logisticregression__solver': 'lbfgs'}
0.8231220657276996


In [6]:

results = grid_search.cv_results_
#딕셔너리 형태라 데이터프레임으로 바꿔보는게 좋음
results_df = pd.DataFrame(results).sort_values("rank_test_score")
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_logisticregression__C,param_logisticregression__class_weight,param_logisticregression__l1_ratio,param_logisticregression__max_iter,param_logisticregression__solver,params,...,split2_train_score,split3_train_score,split4_train_score,split5_train_score,split6_train_score,split7_train_score,split8_train_score,split9_train_score,mean_train_score,std_train_score
22,0.014289,0.003361,0.005610,0.000974,1.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 1, 'logisticregressi...",...,0.815913,0.822153,0.826833,0.819033,0.822153,0.829953,0.823713,0.829953,0.824595,0.004596
20,0.013342,0.002582,0.007221,0.002146,0.500,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.819033,0.822153,0.825273,0.819033,0.822153,0.831513,0.823713,0.831513,0.823970,0.004190
8,0.012459,0.003176,0.006652,0.002642,1.000,NaN,1,10000,liblinear,"{'logisticregression__C': 1, 'logisticregressi...",...,0.811232,0.822153,0.822153,0.814353,0.822153,0.831513,0.822153,0.829953,0.822254,0.005802
26,0.013633,0.002946,0.005810,0.002086,100.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 100, 'logisticregres...",...,0.808112,0.822153,0.825273,0.814353,0.823713,0.836193,0.814353,0.828393,0.823973,0.009090
12,0.025817,0.005716,0.006627,0.002260,100.000,NaN,1,10000,liblinear,"{'logisticregression__C': 100, 'logisticregres...",...,0.808112,0.822153,0.825273,0.814353,0.823713,0.836193,0.814353,0.828393,0.823973,0.009090
10,0.016594,0.004416,0.005086,0.000863,10.000,NaN,1,10000,liblinear,"{'logisticregression__C': 10, 'logisticregress...",...,0.808112,0.822153,0.826833,0.811232,0.822153,0.836193,0.814353,0.828393,0.823661,0.009512
24,0.015274,0.003786,0.007061,0.002171,10.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 10, 'logisticregress...",...,0.808112,0.822153,0.826833,0.811232,0.822153,0.836193,0.815913,0.828393,0.823817,0.009370
6,0.011498,0.002057,0.006094,0.002549,0.500,NaN,1,10000,liblinear,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.812793,0.801872,0.809672,0.803432,0.804992,0.826833,0.828393,0.812793,0.813828,0.008822
18,0.014887,0.004100,0.006057,0.001736,0.100,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.1, 'logisticregres...",...,0.797192,0.804992,0.800312,0.800312,0.797192,0.806552,0.812793,0.804992,0.803059,0.004668
4,0.012428,0.001856,0.006248,0.002371,0.100,NaN,1,10000,liblinear,"{'logisticregression__C': 0.1, 'logisticregres...",...,0.790952,0.798752,0.797192,0.794072,0.794072,0.806552,0.800312,0.806552,0.798846,0.004890


### 최적 모델 저장

In [7]:
import joblib
best_model = grid_search.best_estimator_
joblib.dump(best_model, "../data/model/tuned_logistic_regression.pkl")

['../data/model/tuned_logistic_regression.pkl']